In [204]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [207]:
# Създаване на NeighborLoader
from torch_geometric.loader import NeighborLoader
train_loader = NeighborLoader(
    data,
    input_nodes=data.train_mask,
    num_neighbors=[15, 10],
    batch_size=128,
    shuffle=True
)

In [208]:
# Дефиниране на модела
from torch_geometric.nn import SAGEConv
class GraphSAGE(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = SAGEConv(
            dataset.num_node_features,
            64
        )

        self.conv2 = SAGEConv(
            64,
            dataset.num_classes
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)

        return x

In [209]:
# Обучение
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = GraphSAGE().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

model.train()

for epoch in range(100):

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(
            batch.x,
            batch.edge_index
        )

        loss = F.cross_entropy(
            out[:batch.batch_size],
            batch.y[:batch.batch_size]
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1:3d}  Loss={total_loss:.4f}"
    )

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:
# Проверка на размера на партидите
for batch in train_loader:

    print(batch)

    print("Възли:", batch.num_nodes)
    print("Ребра:", batch.num_edges)

    break

In [212]:
import torch_sparse

ModuleNotFoundError: No module named 'torch_sparse'